In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests

In [ ]:
#Breaker+FVG Signal - Static

#Swings detector 
Ticker = 'SBILIFE.NS'


def telegram_alert(message):
        url = f'https://api.telegram.org/bot8520982329:AAGr_PfQPzUFI2zQDWJ41TAtmCobla2JcZY/sendMessage'
        Data = {'chat_id': '-5245957606',
        'text': message
        }
        requests.post(url,data=Data) 

period = round(((1/7)*-position)+10)

BTC = yf.Ticker(Ticker).history(period=f'{period}d', interval='1h')
if BTC.empty:
            print(f'No data found for {Ticker}')


#CODE - Swing

BTC['Swing_High'] = np.where((BTC['High']>BTC['High'].shift(-1))&(BTC['High']>BTC['High'].shift(1)),"Swing High","NA")
BTC['Swing_Low'] = np.where((BTC['Low']<BTC['Low'].shift(-1))&(BTC['Low']<BTC['Low'].shift(1)),"Swing Low","NA")

BTC['Swing_High_Value'] = np.where(BTC['Swing_High']=="Swing High",BTC['High'],np.nan)
BTC['Swing_Low_Value'] = np.where(BTC['Swing_Low']=="Swing Low",BTC['Low'],np.nan)

BTC['Last_Swing_High'] = BTC['Swing_High_Value'].ffill()
BTC['Last_Swing_Low'] = BTC['Swing_Low_Value'].ffill()

BTC['Prev_Swing_High'] = BTC['Last_Swing_High'].shift(1)
BTC['Prev_Swing_Low'] = BTC['Last_Swing_Low'].shift(1)

conditions = [
        (BTC['Swing_High']=='Swing High')&
            (BTC['Last_Swing_High']>BTC['Prev_Swing_High']),

            (BTC['Swing_High']=='Swing High')&
            (BTC['Last_Swing_High']<BTC['Prev_Swing_High']),

            (BTC['Swing_Low']=='Swing Low')&
            (BTC['Last_Swing_Low']>BTC['Prev_Swing_Low']),

            (BTC['Swing_Low']=='Swing Low')&
            (BTC['Last_Swing_Low']<BTC['Prev_Swing_Low']),
        ]

labels = ['Higher High','Lower High','Higher Low','Lower Low']

BTC['Master_Swing'] = np.select(conditions,labels,default="")

        #Swings pattern detector 


        #Combined Signal (Swing High/Low)
conditions = [
            (BTC['High']>BTC['High'].shift(-1))&(BTC['High']>BTC['High'].shift(1)),

            (BTC['Low']<BTC['Low'].shift(-1))&(BTC['Low']<BTC['Low'].shift(1))
        ] 

signals = ['Swing High','Swing Low']

BTC['Main_Signal'] = np.select(conditions,signals,default="")

Subset_df = BTC[BTC['Main_Signal'] !=''].copy()

Subset_df['Breaker_Signal'] = (
            (Subset_df['Main_Signal'] == 'Swing High')&
            (Subset_df['Main_Signal'].shift(1)=='Swing Low')&
            (Subset_df['Main_Signal'].shift(2)=='Swing High')&
            (Subset_df['Main_Signal'].shift(3)=='Swing Low')&
            (Subset_df['High']>Subset_df['High'].shift(2))&
            (Subset_df['Low'].shift(1)<Subset_df['Low'].shift(3))
            )
BTC["Breaker_Signal"] = False

BTC.loc[Subset_df.index,'Breaker_Signal'] = Subset_df['Breaker_Signal']
        #In BTC using the index of subset df, fill the Breaker signal column in BTC with values of subset_df["breaker signal"]


#CODE - FVG
T0_High = BTC['High']
T0_Low = BTC['Low']
T1_High = BTC['High'].shift(2)
T1_Low = BTC['Low'].shift(2)

conditions = [
    (T0_Low>T1_High),
    (T0_High<T1_Low)
]

Signals = ["Bullish_FVG","Bearish_FVG"]

BTC['FVG_Signal'] = np.select(conditions,Signals,default="NA")

BTC['T0_Bull_Low'] = np.where(BTC['FVG_Signal']=="Bullish_FVG",BTC['Low'],np.nan)
BTC['T1_Bull_High'] = np.where(BTC['FVG_Signal']=="Bullish_FVG",BTC['High'].shift(2),np.nan)

BTC['T0_Bearish_High'] = np.where(BTC['FVG_Signal']=="Bearish_FVG",BTC['High'],np.nan)
BTC['T1_Bearish_Low'] = np.where(BTC['FVG_Signal']=="Bearish_FVG",BTC['Low'].shift(2),np.nan)

Sub_FVG_df = BTC[BTC['FVG_Signal'] !='NA'].copy()


#2

conditions = [
        (Sub_FVG_df['FVG_Signal']== "Bullish_FVG")&
        (Sub_FVG_df['FVG_Signal'].shift(1)== "Bearish_FVG")&
        (Sub_FVG_df['T0_Bull_Low']>Sub_FVG_df['T0_Bearish_High'].shift(1))&
        (Sub_FVG_df['T1_Bearish_Low'].shift(1)>Sub_FVG_df['T1_Bull_High'])

]

Choice = ['FVG_Overlap']

Sub_FVG_df['FVG_Overlap']=np.select(conditions,Choice,default='NA')

BTC['FVG_Overlap_Main'] = False

BTC.loc[Sub_FVG_df.index,'FVG_Overlap_Main'] = Sub_FVG_df['FVG_Overlap']=="FVG_Overlap"

#Storing the index of when FVG overlap occured

Sub_FVG_df['FVG_OL_index'] = Sub_FVG_df.index.to_series().where(
    Sub_FVG_df['FVG_Overlap']=='FVG_Overlap'
)

BTC['FVG_OL_index'] = (
    Sub_FVG_df['FVG_OL_index']
    .reindex(BTC.index)      # align to BTC index
    .ffill()                 # forward fill immediately
)

#Storing the index of when Swing High(T1) of Breaker signal occured

Subset_df['SH_T1'] = Subset_df.index.to_series().shift(2).where(
    Subset_df['Breaker_Signal']==True
)

BTC.loc[Subset_df.index,'SH_T1'] = Subset_df['SH_T1']

BTC['Breaker+FVG Signal'] = (
        (BTC['Breaker_Signal']==True)&
        (BTC['FVG_OL_index']>=BTC['SH_T1'])
)

position = -50

Breaker_FVG_Signal = BTC['Breaker+FVG Signal'].iloc[position]

if Breaker_FVG_Signal == True:
                message = (f'Alert: Breaker+FVG Formation in {Ticker}')
                telegram_alert(message)
else:
            print(f'No alert for {Ticker}')

BTC.to_csv("SUPREME.csv")
print(BTC)



No alert for SBILIFE.NS
                                  Open         High          Low        Close  \
Datetime                                                                        
2026-03-20 09:15:00+05:30  1907.699951  1916.400024  1892.199951  1892.400024   
2026-03-20 10:15:00+05:30  1892.000000  1906.300049  1890.199951  1902.900024   
2026-03-20 11:15:00+05:30  1903.599976  1912.000000  1896.000000  1901.400024   
2026-03-20 12:15:00+05:30  1901.699951  1903.500000  1894.400024  1902.199951   
2026-03-20 13:15:00+05:30  1902.199951  1905.900024  1899.500000  1902.699951   
...                                ...          ...          ...          ...   
2026-04-17 11:15:00+05:30  1967.500000  1971.900024  1964.300049  1968.900024   
2026-04-17 12:15:00+05:30  1969.400024  1971.699951  1967.000000  1970.300049   
2026-04-17 13:15:00+05:30  1970.300049  1973.000000  1967.800049  1969.599976   
2026-04-17 14:15:00+05:30  1969.599976  1975.599976  1969.199951  1972.300049   
2026


KeyboardInterrupt



In [ ]:
#Breaker+FVG Signal - Dynamic

#Swings detector 
Ticker_df = pd.read_csv('NSE_Symbols.csv')


def telegram_alert(message):
        url = f'https://api.telegram.org/bot8520982329:AAGr_PfQPzUFI2zQDWJ41TAtmCobla2JcZY/sendMessage'
        Data = {'chat_id': '-5245957606',
        'text': message
        }
        requests.post(url,data=Data) 


Alert_List = []

for Ticker in Ticker_df['Symbol']:
    position = -21
    period = round(((1/7)*-position)+10)
    BTC = yf.Ticker(Ticker).history(period=f'{period}d', interval='1h')
    if BTC.empty:
            print(f'No data found for {Ticker}')
            continue


    #CODE - Swing

    BTC['Swing_High'] = np.where((BTC['High']>BTC['High'].shift(-1))&(BTC['High']>BTC['High'].shift(1)),"Swing High","NA")
    BTC['Swing_Low'] = np.where((BTC['Low']<BTC['Low'].shift(-1))&(BTC['Low']<BTC['Low'].shift(1)),"Swing Low","NA")

    BTC['Swing_High_Value'] = np.where(BTC['Swing_High']=="Swing High",BTC['High'],np.nan)
    BTC['Swing_Low_Value'] = np.where(BTC['Swing_Low']=="Swing Low",BTC['Low'],np.nan)

    BTC['Last_Swing_High'] = BTC['Swing_High_Value'].ffill()
    BTC['Last_Swing_Low'] = BTC['Swing_Low_Value'].ffill()

    BTC['Prev_Swing_High'] = BTC['Last_Swing_High'].shift(1)
    BTC['Prev_Swing_Low'] = BTC['Last_Swing_Low'].shift(1)

    conditions = [
        (BTC['Swing_High']=='Swing High')&
            (BTC['Last_Swing_High']>BTC['Prev_Swing_High']),

            (BTC['Swing_High']=='Swing High')&
            (BTC['Last_Swing_High']<BTC['Prev_Swing_High']),

            (BTC['Swing_Low']=='Swing Low')&
            (BTC['Last_Swing_Low']>BTC['Prev_Swing_Low']),

            (BTC['Swing_Low']=='Swing Low')&
            (BTC['Last_Swing_Low']<BTC['Prev_Swing_Low']),
        ]

    labels = ['Higher High','Lower High','Higher Low','Lower Low']

    BTC['Master_Swing'] = np.select(conditions,labels,default="")

            #Swings pattern detector 


            #Combined Signal (Swing High/Low)
    conditions = [
                (BTC['High']>BTC['High'].shift(-1))&(BTC['High']>BTC['High'].shift(1)),

                (BTC['Low']<BTC['Low'].shift(-1))&(BTC['Low']<BTC['Low'].shift(1))
            ] 

    signals = ['Swing High','Swing Low']

    BTC['Main_Signal'] = np.select(conditions,signals,default="")

    Subset_df = BTC[BTC['Main_Signal'] !=''].copy()

    Subset_df['Breaker_Signal'] = (
                (Subset_df['Main_Signal'] == 'Swing High')&
                (Subset_df['Main_Signal'].shift(1)=='Swing Low')&
                (Subset_df['Main_Signal'].shift(2)=='Swing High')&
                (Subset_df['Main_Signal'].shift(3)=='Swing Low')&
                (Subset_df['High']>Subset_df['High'].shift(2))&
                (Subset_df['Low'].shift(1)<Subset_df['Low'].shift(3))
                )
    BTC["Breaker_Signal"] = False

    BTC.loc[Subset_df.index,'Breaker_Signal'] = Subset_df['Breaker_Signal']
            #In BTC using the index of subset df, fill the Breaker signal column in BTC with values of subset_df["breaker signal"]


    #CODE - FVG
    T0_High = BTC['High']
    T0_Low = BTC['Low']
    T1_High = BTC['High'].shift(2)
    T1_Low = BTC['Low'].shift(2)

    conditions = [
        (T0_Low>T1_High),
        (T0_High<T1_Low)
    ]

    Signals = ["Bullish_FVG","Bearish_FVG"]

    BTC['FVG_Signal'] = np.select(conditions,Signals,default="NA")

    BTC['T0_Bull_Low'] = np.where(BTC['FVG_Signal']=="Bullish_FVG",BTC['Low'],np.nan)
    BTC['T1_Bull_High'] = np.where(BTC['FVG_Signal']=="Bullish_FVG",BTC['High'].shift(2),np.nan)

    BTC['T0_Bearish_High'] = np.where(BTC['FVG_Signal']=="Bearish_FVG",BTC['High'],np.nan)
    BTC['T1_Bearish_Low'] = np.where(BTC['FVG_Signal']=="Bearish_FVG",BTC['Low'].shift(2),np.nan)

    Sub_FVG_df = BTC[BTC['FVG_Signal'] !='NA'].copy()


    #2

    conditions = [
            (Sub_FVG_df['FVG_Signal']== "Bullish_FVG")&
            (Sub_FVG_df['FVG_Signal'].shift(1)== "Bearish_FVG")&
            (Sub_FVG_df['T0_Bull_Low']>Sub_FVG_df['T0_Bearish_High'].shift(1))&
            (Sub_FVG_df['T1_Bearish_Low'].shift(1)>Sub_FVG_df['T1_Bull_High'])

    ]

    Choice = ['FVG_Overlap']

    Sub_FVG_df['FVG_Overlap']=np.select(conditions,Choice,default='NA')

    BTC['FVG_Overlap_Main'] = False

    BTC.loc[Sub_FVG_df.index,'FVG_Overlap_Main'] = Sub_FVG_df['FVG_Overlap']=="FVG_Overlap"

    #Storing the index of when FVG overlap occured

    Sub_FVG_df['FVG_OL_index'] = Sub_FVG_df.index.to_series().where(
        Sub_FVG_df['FVG_Overlap']=='FVG_Overlap'
    )

    BTC['FVG_OL_index'] = (
        Sub_FVG_df['FVG_OL_index']
        .reindex(BTC.index)      # align to BTC index
        .ffill()                 # forward fill immediately
    )

    #Storing the index of when Swing High(T1) of Breaker signal occured

    Subset_df['SH_T1'] = Subset_df.index.to_series().shift(2).where(
        Subset_df['Breaker_Signal']==True
    )

    BTC.loc[Subset_df.index,'SH_T1'] = Subset_df['SH_T1']

    BTC['Breaker+FVG Signal'] = (
            (BTC['Breaker_Signal']==True)&
            (BTC['FVG_OL_index']>=BTC['SH_T1'])
    )

    Breaker_FVG_Signal = BTC['Breaker+FVG Signal'].iloc[position]

    if Breaker_FVG_Signal == True:
                # Append both ticker and timestamp (index) as a string
                alert_time = BTC.index[position]
                Alert_List.append(f"{Ticker} at {alert_time}")  
    else:
        print(f'No alert for {Ticker}')

if Alert_List:
        message = (
                "Breaker+FVG Signal in the following:\n\n"
                + f"Candle: {position}\n\n"
                +"\n".join(Alert_List) #Convert the list into string and print them each on new line \n 
        )
        telegram_alert(message)




No alert for AARTIIND.NS
No alert for ABB.NS
No alert for ABBOTINDIA.NS
No alert for ABFRL.NS
No alert for ACC.NS
No alert for ADANIENT.NS
No alert for ADANIPORTS.NS
No alert for ALKEM.NS
No alert for AMBUJACEM.NS
No alert for APOLLOHOSP.NS
No alert for APOLLOTYRE.NS
No alert for ASHOKLEY.NS
No alert for ASIANPAINT.NS
No alert for ASTRAL.NS
No alert for ATUL.NS
No alert for AUBANK.NS
No alert for AUROPHARMA.NS
No alert for BAJAJ-AUTO.NS
No alert for BAJFINANCE.NS
No alert for BALKRISIND.NS
No alert for BALRAMCHIN.NS
No alert for BANDHANBNK.NS
No alert for BANKBARODA.NS
No alert for BATAINDIA.NS
No alert for BEL.NS
No alert for BERGEPAINT.NS
No alert for BHEL.NS
No alert for BIOCON.NS
No alert for BOSCHLTD.NS
No alert for BPCL.NS
No alert for BRITANNIA.NS
No alert for BSOFT.NS
No alert for CANBK.NS
No alert for CANFINHOME.NS
No alert for CHAMBLFERT.NS
No alert for CHOLAFIN.NS
No alert for CIPLA.NS
No alert for COALINDIA.NS
No alert for COFORGE.NS
No alert for COLPAL.NS
No alert for CONC

In [ ]:
#Breaker+FVG Signal - Dynamic + Loop for back testing


position_df = pd.read_csv("position.csv")

for position in position_df['Count']:
        print(f"Processing Candle{position}")

        #Swings detector 
        Ticker_df = pd.read_csv('NSE_Symbols.csv')
        
        def telegram_alert(message):
            url = f'https://api.telegram.org/bot8520982329:AAGr_PfQPzUFI2zQDWJ41TAtmCobla2JcZY/sendMessage'
            Data = {'chat_id': '-5245957606',
            'text': message
            }
            requests.post(url,data=Data) 


        Alert_List = []

        for Ticker in Ticker_df['Symbol']:
                period = round(((1/7)*-position)+10)
                BTC = yf.Ticker(Ticker).history(period=f'{period}d', interval='1h')
                if BTC.empty:
                        continue


                #CODE - Swing

                BTC['Swing_High'] = np.where((BTC['High']>BTC['High'].shift(-1))&(BTC['High']>BTC['High'].shift(1)),"Swing High","NA")
                BTC['Swing_Low'] = np.where((BTC['Low']<BTC['Low'].shift(-1))&(BTC['Low']<BTC['Low'].shift(1)),"Swing Low","NA")

                BTC['Swing_High_Value'] = np.where(BTC['Swing_High']=="Swing High",BTC['High'],np.nan)
                BTC['Swing_Low_Value'] = np.where(BTC['Swing_Low']=="Swing Low",BTC['Low'],np.nan)

                BTC['Last_Swing_High'] = BTC['Swing_High_Value'].ffill()
                BTC['Last_Swing_Low'] = BTC['Swing_Low_Value'].ffill()

                BTC['Prev_Swing_High'] = BTC['Last_Swing_High'].shift(1)
                BTC['Prev_Swing_Low'] = BTC['Last_Swing_Low'].shift(1)

                conditions = [
                (BTC['Swing_High']=='Swing High')&
                        (BTC['Last_Swing_High']>BTC['Prev_Swing_High']),

                        (BTC['Swing_High']=='Swing High')&
                        (BTC['Last_Swing_High']<BTC['Prev_Swing_High']),

                        (BTC['Swing_Low']=='Swing Low')&
                        (BTC['Last_Swing_Low']>BTC['Prev_Swing_Low']),

                        (BTC['Swing_Low']=='Swing Low')&
                        (BTC['Last_Swing_Low']<BTC['Prev_Swing_Low']),
                ]

                labels = ['Higher High','Lower High','Higher Low','Lower Low']

                BTC['Master_Swing'] = np.select(conditions,labels,default="")

                        #Swings pattern detector 


                        #Combined Signal (Swing High/Low)
                conditions = [
                        (BTC['High']>BTC['High'].shift(-1))&(BTC['High']>BTC['High'].shift(1)),

                        (BTC['Low']<BTC['Low'].shift(-1))&(BTC['Low']<BTC['Low'].shift(1))
                        ] 

                signals = ['Swing High','Swing Low']

                BTC['Main_Signal'] = np.select(conditions,signals,default="")

                Subset_df = BTC[BTC['Main_Signal'] !=''].copy()

                Subset_df['Breaker_Signal'] = (
                        (Subset_df['Main_Signal'] == 'Swing High')&
                        (Subset_df['Main_Signal'].shift(1)=='Swing Low')&
                        (Subset_df['Main_Signal'].shift(2)=='Swing High')&
                        (Subset_df['Main_Signal'].shift(3)=='Swing Low')&
                        (Subset_df['High']>Subset_df['High'].shift(2))&
                        (Subset_df['Low'].shift(1)<Subset_df['Low'].shift(3))
                        )
                BTC["Breaker_Signal"] = False

                BTC.loc[Subset_df.index,'Breaker_Signal'] = Subset_df['Breaker_Signal']
                        #In BTC using the index of subset df, fill the Breaker signal column in BTC with values of subset_df["breaker signal"]


                #CODE - FVG
                T0_High = BTC['High']
                T0_Low = BTC['Low']
                T1_High = BTC['High'].shift(2)
                T1_Low = BTC['Low'].shift(2)

                conditions = [
                (T0_Low>T1_High),
                (T0_High<T1_Low)
                ]

                Signals = ["Bullish_FVG","Bearish_FVG"]

                BTC['FVG_Signal'] = np.select(conditions,Signals,default="NA")

                BTC['T0_Bull_Low'] = np.where(BTC['FVG_Signal']=="Bullish_FVG",BTC['Low'],np.nan)
                BTC['T1_Bull_High'] = np.where(BTC['FVG_Signal']=="Bullish_FVG",BTC['High'].shift(2),np.nan)

                BTC['T0_Bearish_High'] = np.where(BTC['FVG_Signal']=="Bearish_FVG",BTC['High'],np.nan)
                BTC['T1_Bearish_Low'] = np.where(BTC['FVG_Signal']=="Bearish_FVG",BTC['Low'].shift(2),np.nan)

                Sub_FVG_df = BTC[BTC['FVG_Signal'] !='NA'].copy()


                #2

                conditions = [
                        (Sub_FVG_df['FVG_Signal']== "Bullish_FVG")&
                        (Sub_FVG_df['FVG_Signal'].shift(1)== "Bearish_FVG")&
                        (Sub_FVG_df['T0_Bull_Low']>Sub_FVG_df['T0_Bearish_High'].shift(1))&
                        (Sub_FVG_df['T1_Bearish_Low'].shift(1)>Sub_FVG_df['T1_Bull_High'])

                ]

                Choice = ['FVG_Overlap']

                Sub_FVG_df['FVG_Overlap']=np.select(conditions,Choice,default='NA')

                BTC['FVG_Overlap_Main'] = False

                BTC.loc[Sub_FVG_df.index,'FVG_Overlap_Main'] = Sub_FVG_df['FVG_Overlap']=="FVG_Overlap"

                #Storing the index of when FVG overlap occured

                Sub_FVG_df['FVG_OL_index'] = Sub_FVG_df.index.to_series().where(
                Sub_FVG_df['FVG_Overlap']=='FVG_Overlap'
                )

                BTC['FVG_OL_index'] = (
                Sub_FVG_df['FVG_OL_index']
                .reindex(BTC.index)      # align to BTC index
                .ffill()                 # forward fill immediately
                )

                #Storing the index of when Swing High(T1) of Breaker signal occured

                Subset_df['SH_T1'] = Subset_df.index.to_series().shift(2).where(
                Subset_df['Breaker_Signal']==True
                )

                BTC.loc[Subset_df.index,'SH_T1'] = Subset_df['SH_T1']

                BTC['Breaker+FVG Signal'] = (
                        (BTC['Breaker_Signal']==True)&
                        (BTC['FVG_OL_index']>=BTC['SH_T1'])
                )

                
                Breaker_FVG_Signal = BTC['Breaker+FVG Signal'].iloc[position]
                
                if Breaker_FVG_Signal == True:
                        alert_time = BTC.index[position]
                        Alert_List.append(f"{Ticker} at {alert_time}")  #Add the ticker to Alert List
                        

        if Alert_List:
            message = (
                    "Breaker+FVG Signal in the following:\n\n"
                    + f"Candle: {position}\n\n"
                    +"\n".join(Alert_List) #Convert the list into string and print them each on new line \n 
            )
            telegram_alert(message)




In [3]:
#Breaker+FVG Signal - Dynamic (Improved) - Accumulated Liquidity v2
print("Running: Dynamic Improved + Accumulated Liquidity v2")

Ticker_df    = pd.read_csv('NSE_Symbols.csv')
tickers_list = Ticker_df['Symbol'].tolist()

def telegram_alert(message):
    url = 'https://api.telegram.org/bot8520982329:AAGr_PfQPzUFI2zQDWJ41TAtmCobla2JcZY/sendMessage'
    requests.post(url, data={'chat_id': '-5245957606', 'text': message})

position_input = -108  # use 112 or -112 to scan 112 candles back from the latest candle
position       = -abs(position_input)
position_label = abs(position_input)
ATR_BASELINE = 14   # candles before T2 used as volatility baseline (configurable)
ISL_LOOKBACK = 30   # extra calendar days fetched purely for ISL/ISH history (no effect on position logic)
LIQ_ISL_MAX_ATR_DISTANCE = 3.0  # lower ISL base must be within this many ATRs of T3
LIQ_BREACH_TOLERANCE_ATR = 0.05  # ignore tiny pre-sweep undercuts when deciding if a swing low stayed protected
period       = round(((1/7) * -position) + 10)
fetch_period = period + ISL_LOOKBACK   # larger window used for download; position is always from END so this is safe

def mean_atr(df):
    """Mean True Range over a DataFrame slice."""
    prev_c = df['Close'].shift(1).fillna(df['Close'].iloc[0])
    tr = pd.concat([
        df['High'] - df['Low'],
        (df['High'] - prev_c).abs(),
        (df['Low']  - prev_c).abs()
    ], axis=1).max(axis=1)
    return tr.mean()

def accumulated_liquidity(BTC, Subset_df, t3_ts, t1_ts, t3_low, t1_low, atr_baseline):
    """Profile accumulated liquidity below protected swing lows before the breaker sweep."""
    prior_isls = BTC['ISL_Price'].loc[:t3_ts].dropna()
    prior_isls = prior_isls[prior_isls < t3_low]
    tolerance = atr_baseline * LIQ_BREACH_TOLERANCE_ATR if atr_baseline > 0 else 0.0

    if atr_baseline > 0:
        prior_isls = prior_isls[(t3_low - prior_isls) <= (LIQ_ISL_MAX_ATR_DISTANCE * atr_baseline)]

    if prior_isls.empty:
        return 0, float('nan'), False, 0.0, 0, 0.0, 0, 0.0

    lowest_isl = prior_isls.min()
    base_isl_ts = prior_isls[prior_isls == lowest_isl].index[-1]
    base_isl = float(lowest_isl)

    prior_swing_lows = Subset_df[
        (Subset_df.index >= base_isl_ts) &
        (Subset_df.index < t3_ts) &
        (Subset_df['Main_Signal'] == 'Swing Low')
    ]

    protected_levels = []
    hold_bars = []
    defense_touches = 0
    volume_scores = []
    has_volume = 'Volume' in BTC.columns
    avg_volume = BTC['Volume'].loc[base_isl_ts:t3_ts].replace(0, np.nan).mean() if has_volume else np.nan

    for sl_ts, row in prior_swing_lows.iterrows():
        sl_price = row['Low']
        later_swing_lows = prior_swing_lows[(prior_swing_lows.index > sl_ts) & (prior_swing_lows.index < t1_ts)]
        breached_before_sweep = (later_swing_lows['Low'] < (sl_price - tolerance)).any()
        swept_at_t1 = t1_low < (sl_price - tolerance)

        if not breached_before_sweep and swept_at_t1:
            protected_levels.append(float(sl_price))
            hold_bars.append(BTC.index.get_loc(t1_ts) - BTC.index.get_loc(sl_ts))

            defend_window = BTC[(BTC.index > sl_ts) & (BTC.index < t1_ts)]
            near_level = defend_window['Low'].between(sl_price - tolerance, sl_price + tolerance)
            defense_touches += int(near_level.sum())

            if has_volume and avg_volume > 0:
                sl_iloc = BTC.index.get_loc(sl_ts)
                v_start = max(0, sl_iloc - 1)
                v_end = min(len(BTC), sl_iloc + 2)
                swing_vol = BTC['Volume'].iloc[v_start:v_end].replace(0, np.nan).mean()
                if not np.isnan(swing_vol):
                    volume_scores.append(swing_vol / avg_volume)

    protected_lows = len(protected_levels)
    base_isl_swept = t1_low < base_isl
    avg_hold_bars = round(float(np.mean(hold_bars)), 2) if hold_bars else 0.0
    volume_score = round(float(np.mean(volume_scores)), 3) if volume_scores else 0.0

    protected_levels = sorted(protected_levels)
    cluster_count = 0
    if protected_levels:
        cluster_count = 1
        for prev_level, level in zip(protected_levels, protected_levels[1:]):
            if abs(level - prev_level) > tolerance:
                cluster_count += 1

    spacing_atr = 0.0
    if len(protected_levels) > 1 and atr_baseline > 0:
        spacing_atr = round(float(np.mean(np.diff(protected_levels)) / atr_baseline), 3)

    return protected_lows, base_isl, base_isl_swept, avg_hold_bars, defense_touches, volume_score, cluster_count, spacing_atr

def unresolved_deeper_isl(BTC, t3_ts, t1_low, atr_baseline):
    """Find historical ISL liquidity below T1 that the current breaker sweep did not take."""
    historical_isls = BTC['ISL_Price'].loc[:t3_ts].dropna()
    deeper_isls = historical_isls[historical_isls < t1_low]

    if deeper_isls.empty:
        return False, float('nan'), float('nan'), 0

    nearest_deeper_isl = float(deeper_isls.max())
    distance_atr = round((t1_low - nearest_deeper_isl) / atr_baseline, 3) if atr_baseline > 0 else float('nan')
    return True, nearest_deeper_isl, distance_atr, len(deeper_isls)

def target_liquidity(BTC, Subset_df, idx, idx_high, atr_baseline):
    """Profile buy-side target liquidity from a base ISH to the breaker confirmation high."""
    tolerance = atr_baseline * LIQ_BREACH_TOLERANCE_ATR if atr_baseline > 0 else 0.0
    prior_ish = BTC['ISH_Price'].loc[:idx].dropna()
    prior_ish = prior_ish[prior_ish > idx_high]

    if prior_ish.empty:
        return 0, float('nan'), False, float('nan'), 0.0, 0, 0.0, 0, 0.0

    base_ish = float(prior_ish.min())
    base_ish_ts = prior_ish[prior_ish == base_ish].index[-1]
    target_distance_atr = round((base_ish - idx_high) / atr_baseline, 3) if atr_baseline > 0 else float('nan')

    prior_swing_highs = Subset_df[
        (Subset_df.index >= base_ish_ts) &
        (Subset_df.index < idx) &
        (Subset_df['Main_Signal'] == 'Swing High')
    ]

    target_levels = []
    hold_bars = []
    target_touches = 0
    volume_scores = []
    has_volume = 'Volume' in BTC.columns
    avg_volume = BTC['Volume'].loc[base_ish_ts:idx].replace(0, np.nan).mean() if has_volume else np.nan

    for sh_ts, row in prior_swing_highs.iterrows():
        sh_price = row['High']
        later_swing_highs = prior_swing_highs[(prior_swing_highs.index > sh_ts) & (prior_swing_highs.index < idx)]
        breached_before_signal = (later_swing_highs['High'] > (sh_price + tolerance)).any()
        still_above_breaker = sh_price > (idx_high + tolerance)

        if still_above_breaker and not breached_before_signal:
            target_levels.append(float(sh_price))
            hold_bars.append(BTC.index.get_loc(idx) - BTC.index.get_loc(sh_ts))

            defend_window = BTC[(BTC.index > sh_ts) & (BTC.index < idx)]
            near_level = defend_window['High'].between(sh_price - tolerance, sh_price + tolerance)
            target_touches += int(near_level.sum())

            if has_volume and avg_volume > 0:
                sh_iloc = BTC.index.get_loc(sh_ts)
                v_start = max(0, sh_iloc - 1)
                v_end = min(len(BTC), sh_iloc + 2)
                swing_vol = BTC['Volume'].iloc[v_start:v_end].replace(0, np.nan).mean()
                if not np.isnan(swing_vol):
                    volume_scores.append(swing_vol / avg_volume)

    target_count = len(target_levels)
    base_ish_swept = idx_high > base_ish
    avg_hold_bars = round(float(np.mean(hold_bars)), 2) if hold_bars else 0.0
    volume_score = round(float(np.mean(volume_scores)), 3) if volume_scores else 0.0

    target_levels = sorted(target_levels)
    cluster_count = 0
    if target_levels:
        cluster_count = 1
        for prev_level, level in zip(target_levels, target_levels[1:]):
            if abs(level - prev_level) > tolerance:
                cluster_count += 1

    spacing_atr = 0.0
    if len(target_levels) > 1 and atr_baseline > 0:
        spacing_atr = round(float(np.mean(np.diff(target_levels)) / atr_baseline), 3)

    return target_count, base_ish, base_ish_swept, target_distance_atr, avg_hold_bars, target_touches, volume_score, cluster_count, spacing_atr

# ── Single bulk download ──────────────────────────────────────────────────────
# fetch_period includes ISL_LOOKBACK extra days so ISL/ISH have sufficient swing history.
# BTC.index[position] is always relative to the END — extra leading history is transparent.
raw_data    = yf.download(tickers_list, period=f'{fetch_period}d', interval='1h',
                          group_by='ticker', auto_adjust=True, progress=False)
multi_index = isinstance(raw_data.columns, pd.MultiIndex)

# (Ticker, ratio, atr_ratio, fvg_atr, gap_eff, drop_fvg, score, isl_sweep, isl_level, protected_lows, liq_base_isl, base_isl_swept, avg_hold_bars, defense_touches, volume_score, cluster_count, spacing_atr, deeper_isl_pending, nearest_deeper_isl, deeper_isl_atr, deeper_isl_count, target_count, base_ish, base_ish_swept, target_distance_atr, target_hold_bars, target_touches, target_volume_score, target_cluster_count, target_spacing_atr)
#   ratio     = drop / rise              — depth of retracement vs initial leg
#   atr_ratio = atr_downswing / atr_base — candle energy during drop vs normal
#   fvg_atr   = total_fvg_size / atr_base — FVG size relative to ticker volatility
#   gap_eff   = mean(gap / driver_candle_range) — avg fraction of driving candle captured as gap (display only)
#   drop_fvg  = drop / total_fvg_size    — how many times larger drop is vs total FVG gap (display only)
#   score     = ratio × atr_ratio × fvg_atr
#   isl_sweep = True if T3 (SL) was above active ISL and T1 (SL) swept below it (display only)
#   isl_level = the ISL price level that was swept (display only, only when isl_sweep is True)
#   protected_lows = prior protected swing lows between lower ISL base and T3 that T1 finally swept
#   liq_base_isl = lower ISL base used for accumulated-liquidity count
#   base_isl_swept = True if T1 also swept below liq_base_isl
#   avg_hold_bars = average candles each protected low survived before sweep
#   defense_touches = candles that retested/defended protected lows before sweep
#   volume_score = mean swing-low volume vs local average volume
#   cluster_count = number of distinct protected-low price clusters
#   spacing_atr = average spacing between protected lows in ATR units; lower means tighter compression
#   deeper_isl_pending = True if historical ISL liquidity remains below T1 after the sweep
#   nearest_deeper_isl = closest unresolved deeper ISL below T1
#   deeper_isl_atr = distance from T1 low to nearest_deeper_isl in ATR units
#   deeper_isl_count = number of historical unresolved ISLs below T1
#   target_count = unbreached swing highs from base ISH to breaker confirmation high
#   base_ish = nearest prior ISH above the breaker confirmation high, used as buy-side target base
#   base_ish_swept = True if breaker confirmation high already swept base_ish
#   target_distance_atr = distance from breaker high to base_ish in ATR units
#   target_* metrics mirror liquidity diagnostics for pending buy-side targets from base_ish onward
Alert_List    = []
alert_time    = None
target_time   = None
scan_count    = 0   # tickers with enough data
breaker_hits  = 0   # tickers with a Breaker+FVG signal anywhere in dataset
position_hits = 0   # tickers with Breaker+FVG signal at exactly `position`

for Ticker in tickers_list:
    try:
        BTC = (raw_data[Ticker] if multi_index else raw_data).copy()
    except KeyError:
        continue

    BTC.dropna(how='all', inplace=True)
    if BTC.empty or len(BTC) < abs(position):
        continue

    scan_count += 1
    if target_time is None:
        target_time = BTC.index[position]

    # ── Swing detection ───────────────────────────────────────────────────────
    BTC['Swing_High'] = np.where(
        (BTC['High'] > BTC['High'].shift(-1)) & (BTC['High'] > BTC['High'].shift(1)),
        'Swing High', 'NA')
    BTC['Swing_Low'] = np.where(
        (BTC['Low'] < BTC['Low'].shift(-1)) & (BTC['Low'] < BTC['Low'].shift(1)),
        'Swing Low', 'NA')

    BTC['Swing_High_Value'] = np.where(BTC['Swing_High'] == 'Swing High', BTC['High'], np.nan)
    BTC['Swing_Low_Value']  = np.where(BTC['Swing_Low']  == 'Swing Low',  BTC['Low'],  np.nan)

    BTC['Last_Swing_High'] = BTC['Swing_High_Value'].ffill()
    BTC['Last_Swing_Low']  = BTC['Swing_Low_Value'].ffill()
    BTC['Prev_Swing_High'] = BTC['Last_Swing_High'].shift(1)
    BTC['Prev_Swing_Low']  = BTC['Last_Swing_Low'].shift(1)

    BTC['Master_Swing'] = np.select(
        [(BTC['Swing_High'] == 'Swing High') & (BTC['Last_Swing_High'] > BTC['Prev_Swing_High']),
         (BTC['Swing_High'] == 'Swing High') & (BTC['Last_Swing_High'] < BTC['Prev_Swing_High']),
         (BTC['Swing_Low']  == 'Swing Low')  & (BTC['Last_Swing_Low']  > BTC['Prev_Swing_Low']),
         (BTC['Swing_Low']  == 'Swing Low')  & (BTC['Last_Swing_Low']  < BTC['Prev_Swing_Low'])],
        ['Higher High', 'Lower High', 'Higher Low', 'Lower Low'], default='')

    BTC['Main_Signal'] = np.select(
        [(BTC['High'] > BTC['High'].shift(-1)) & (BTC['High'] > BTC['High'].shift(1)),
         (BTC['Low']  < BTC['Low'].shift(-1))  & (BTC['Low']  < BTC['Low'].shift(1))],
        ['Swing High', 'Swing Low'], default='')

    Subset_df = BTC[BTC['Main_Signal'] != ''].copy()
    if len(Subset_df) < 4:
        continue

    # ── ISL / ISH computation ─────────────────────────────────────────────────
    # Intermediate Swing Low (ISL): a swing low that is lower than both the
    # immediately preceding AND immediately following swing low in the swing series —
    # a local minimum of consecutive swing lows.
    # Forward-filled so every bar carries the last confirmed ISL support level.
    # The extra ISL_LOOKBACK days in fetch_period ensure enough swing history exists
    # for this pattern to form well before T3.
    _sl_s       = Subset_df.loc[Subset_df['Main_Signal'] == 'Swing Low', 'Low']
    _isl_mask   = (_sl_s < _sl_s.shift(1)) & (_sl_s < _sl_s.shift(-1))
    _isl_prices = _sl_s.where(_isl_mask)

    _sh_s       = Subset_df.loc[Subset_df['Main_Signal'] == 'Swing High', 'High']
    _ish_mask   = (_sh_s > _sh_s.shift(1)) & (_sh_s > _sh_s.shift(-1))
    _ish_prices = _sh_s.where(_ish_mask)

    BTC['ISL_Price'] = np.nan
    BTC.loc[_isl_prices.index, 'ISL_Price'] = _isl_prices.values
    BTC['ISL_Price_Ffill'] = BTC['ISL_Price'].ffill()

    BTC['ISH_Price'] = np.nan
    BTC.loc[_ish_prices.index, 'ISH_Price'] = _ish_prices.values
    BTC['ISH_Price_Ffill'] = BTC['ISH_Price'].ffill()

    # ── Breaker signal ────────────────────────────────────────────────────────
    Subset_df['Breaker_Signal'] = (
        (Subset_df['Main_Signal'] == 'Swing High') &
        (Subset_df['Main_Signal'].shift(1) == 'Swing Low') &
        (Subset_df['Main_Signal'].shift(2) == 'Swing High') &
        (Subset_df['Main_Signal'].shift(3) == 'Swing Low') &
        (Subset_df['High'] > Subset_df['High'].shift(2)) &
        (Subset_df['Low'].shift(1) < Subset_df['Low'].shift(3))
    )
    BTC['Breaker_Signal'] = False
    BTC.loc[Subset_df.index, 'Breaker_Signal'] = Subset_df['Breaker_Signal']

    # ── FVG detection ─────────────────────────────────────────────────────────
    BTC['FVG_Signal'] = np.select(
        [(BTC['Low'] > BTC['High'].shift(2)),
         (BTC['High'] < BTC['Low'].shift(2))],
        ['Bullish_FVG', 'Bearish_FVG'], default='NA')

    BTC['T0_Bull_Low']     = np.where(BTC['FVG_Signal'] == 'Bullish_FVG', BTC['Low'],           np.nan)
    BTC['T1_Bull_High']    = np.where(BTC['FVG_Signal'] == 'Bullish_FVG', BTC['High'].shift(2), np.nan)
    BTC['T0_Bearish_High'] = np.where(BTC['FVG_Signal'] == 'Bearish_FVG', BTC['High'],          np.nan)
    BTC['T1_Bearish_Low']  = np.where(BTC['FVG_Signal'] == 'Bearish_FVG', BTC['Low'].shift(2),  np.nan)
    BTC['Mid_Bear_Range']  = np.where(BTC['FVG_Signal'] == 'Bearish_FVG',
                                      BTC['High'].shift(1) - BTC['Low'].shift(1), np.nan)

    Sub_FVG_df = BTC[BTC['FVG_Signal'] != 'NA'].copy()
    Sub_FVG_df['Bear_FVG_ts'] = Sub_FVG_df.index.to_series().shift(1)

    Sub_FVG_df['FVG_Overlap'] = np.select(
        [(Sub_FVG_df['FVG_Signal'] == 'Bullish_FVG') &
         (Sub_FVG_df['FVG_Signal'].shift(1) == 'Bearish_FVG') &
         (Sub_FVG_df['T0_Bull_Low'] > Sub_FVG_df['T0_Bearish_High'].shift(1)) &
         (Sub_FVG_df['T1_Bearish_Low'].shift(1) > Sub_FVG_df['T1_Bull_High'])],
        ['FVG_Overlap'], default='NA')

    fvg_overlap_df = Sub_FVG_df.loc[Sub_FVG_df['FVG_Overlap'] == 'FVG_Overlap', ['Bear_FVG_ts']]

    # ── Confluence + scoring ──────────────────────────────────────────────────
    BTC['Breaker+FVG Signal'] = False

    for idx in Subset_df.index[Subset_df['Breaker_Signal']]:
        pos = Subset_df.index.get_loc(idx)
        if pos < 3:
            continue
        t3_ts = Subset_df.index[pos - 3]
        t2_ts = Subset_df.index[pos - 2]

        cands = fvg_overlap_df[
            (fvg_overlap_df.index          >= t2_ts) &
            (fvg_overlap_df.index          <= idx)   &
            (fvg_overlap_df['Bear_FVG_ts'] >= t2_ts)
        ]
        if len(cands) == 0:
            continue

        BTC.loc[idx, 'Breaker+FVG Signal'] = True
        breaker_hits += 1

        if idx == BTC.index[position]:
            position_hits += 1
            t1_ts   = Subset_df.index[pos - 1]
            t2_high = Subset_df.iloc[pos - 2]['High']
            t3_low  = Subset_df.iloc[pos - 3]['Low']
            t1_low  = Subset_df.iloc[pos - 1]['Low']

            rise  = t2_high - t3_low
            drop  = t2_high - t1_low
            ratio = round(drop / rise, 3) if rise > 0 else 0.0

            # ── ATR ratio ────────────────────────────────────────────────────
            t2_iloc = BTC.index.get_loc(t2_ts)
            t1_iloc = BTC.index.get_loc(t1_ts)

            b_start      = max(0, t2_iloc - ATR_BASELINE + 1)
            atr_baseline = mean_atr(BTC.iloc[b_start : t2_iloc + 1])

            ctx     = max(0, t2_iloc - 1)
            ds_full = BTC.iloc[ctx : t1_iloc + 1]
            prev_c  = ds_full['Close'].shift(1).fillna(ds_full['Close'].iloc[0])
            tr_all  = pd.concat([
                          ds_full['High'] - ds_full['Low'],
                          (ds_full['High'] - prev_c).abs(),
                          (ds_full['Low']  - prev_c).abs()
                      ], axis=1).max(axis=1)
            tr_ds         = tr_all.iloc[1:] if ctx < t2_iloc else tr_all
            atr_downswing = tr_ds.mean() if len(tr_ds) > 0 else atr_baseline
            atr_ratio     = round(atr_downswing / atr_baseline, 3) if atr_baseline > 0 else 0.0

            # ── Bearish FVG size ──────────────────────────────────────────────
            bear_in_window = Sub_FVG_df[
                (Sub_FVG_df.index >= t2_ts) &
                (Sub_FVG_df.index <= idx) &
                (Sub_FVG_df['FVG_Signal'] == 'Bearish_FVG')
            ]
            fvg_size = bear_in_window.apply(
                lambda r: max(r['T1_Bearish_Low'] - r['T0_Bearish_High'], 0.0), axis=1
            ).sum()

            fvg_atr  = round(fvg_size / atr_baseline, 3) if atr_baseline > 0 else 0.0
            drop_fvg = round(drop / fvg_size, 3)         if fvg_size > 0     else 0.0
            gap_eff  = round(bear_in_window.apply(
                lambda r: (max(r['T1_Bearish_Low'] - r['T0_Bearish_High'], 0.0) / r['Mid_Bear_Range'])
                          if r['Mid_Bear_Range'] > 0 else 0.0, axis=1
            ).mean(), 3) if len(bear_in_window) > 0 else 0.0

            score = round(ratio * atr_ratio * fvg_atr, 4)

            # ── ISL Sweep check ───────────────────────────────────────────────
            _isl_hist  = BTC['ISL_Price_Ffill'].loc[:t3_ts].dropna()
            _isl_level = float(_isl_hist.iloc[-1]) if len(_isl_hist) > 0 else float('nan')
            isl_sweep  = (
                not (isinstance(_isl_level, float) and _isl_level != _isl_level) and
                t3_low > _isl_level and
                t1_low < _isl_level
            )

            # ── Accumulated liquidity check ─────────────────────────────────
            protected_lows, liq_base_isl, base_isl_swept, avg_hold_bars, defense_touches, volume_score, cluster_count, spacing_atr = accumulated_liquidity(
                BTC, Subset_df, t3_ts, t1_ts, t3_low, t1_low, atr_baseline
            )
            deeper_isl_pending, nearest_deeper_isl, deeper_isl_atr, deeper_isl_count = unresolved_deeper_isl(
                BTC, t3_ts, t1_low, atr_baseline
            )
            target_count, base_ish, base_ish_swept, target_distance_atr, target_hold_bars, target_touches, target_volume_score, target_cluster_count, target_spacing_atr = target_liquidity(
                BTC, Subset_df, idx, BTC.loc[idx, 'High'], atr_baseline
            )

            if alert_time is None:
                alert_time = BTC.index[position]
            Alert_List.append((Ticker, ratio, atr_ratio, fvg_atr, gap_eff, drop_fvg, score, isl_sweep, _isl_level, protected_lows, liq_base_isl, base_isl_swept, avg_hold_bars, defense_touches, volume_score, cluster_count, spacing_atr, deeper_isl_pending, nearest_deeper_isl, deeper_isl_atr, deeper_isl_count, target_count, base_ish, base_ish_swept, target_distance_atr, target_hold_bars, target_touches, target_volume_score, target_cluster_count, target_spacing_atr))

    # Fallback guard
    if BTC['Breaker+FVG Signal'].iloc[position] and not any(t == Ticker for t, *_ in Alert_List):
        if alert_time is None:
            alert_time = BTC.index[position]
        Alert_List.append((Ticker, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, False, float('nan'), 0, float('nan'), False, 0.0, 0, 0.0, 0, 0.0, False, float('nan'), float('nan'), 0, 0, float('nan'), False, float('nan'), 0.0, 0, 0.0, 0, 0.0))

# ── Scan summary ──────────────────────────────────────────────────────────────
target_time_ist = target_time.tz_convert('Asia/Kolkata') if target_time is not None else target_time
print("Mode: Dynamic Improved + Accumulated Liquidity v2 diagnostics")
print(f"Scan complete — tickers processed: {scan_count} | Breaker+FVG hits (any candle): {breaker_hits} | hits at candle {position_label} back: {position_hits}")
print(f"Candle {position_label} back timestamp: {target_time_ist}")

# ── Rank by score descending ──────────────────────────────────────────────────
Alert_List.sort(key=lambda x: x[6], reverse=True)

# ── Send alert ────────────────────────────────────────────────────────────────
if Alert_List:
    ranked_lines = "\n".join(
        f"{rank}. {ticker}{f' [+ISL Sweep @ {isl_lvl:.2f}]' if isl_sw else ''}{f' [+Base ISL Sweep @ {liq_base:.2f}]' if base_isl_sw else ''}{f' [Deeper ISL pending @ {deep_isl:.2f}, {deep_atr:.2f} ATR]' if deep_pending else ''}{f' [Target ISH @ {base_ish:.2f}, {tgt_dist:.2f} ATR]' if tgt_count > 0 else ' [No clear target ISH]'}  "
        f"(swept_lows: {protected_lows} | base_isl: {liq_base:.2f} | low_hold: {avg_hold:.1f} | low_touches: {touches} | low_vol: {vol_score:.2f}x | low_clusters: {clusters} | deeper_isls: {deep_count} | target_highs: {tgt_count} | base_ish: {base_ish:.2f} | target_hold: {tgt_hold:.1f} | target_touches: {tgt_touches} | target_vol: {tgt_vol:.2f}x | target_clusters: {tgt_clusters} | target_spacing_atr: {tgt_spacing:.3f} | ratio: {ratio:.3f} | atr_r: {atr_r:.3f} | fvg/atr: {fa:.3f} | score: {score:.4f})"
        for rank, (ticker, ratio, atr_r, fa, ge, df, score, isl_sw, isl_lvl, protected_lows, liq_base, base_isl_sw, avg_hold, touches, vol_score, clusters, spacing, deep_pending, deep_isl, deep_atr, deep_count, tgt_count, base_ish, base_ish_sw, tgt_dist, tgt_hold, tgt_touches, tgt_vol, tgt_clusters, tgt_spacing) in enumerate(Alert_List, start=1)
    )
    alert_time_ist = alert_time.tz_convert('Asia/Kolkata') if alert_time is not None else alert_time
    message = (
        "Breaker+FVG signals\n\n"
        f"Candle: {position_label} back\n"
        f"Time: {alert_time_ist}\n\n"
        + ranked_lines
    )
    telegram_alert(message)
    print(message)
else:
    print(f"No Breaker+FVG signals at candle {position_label} back ({target_time_ist}).")
    print(f"  → {breaker_hits} Breaker+FVG patterns exist elsewhere in the data window.")
    print(f"     Try position closer to 0 (e.g. -7, -14, -21) or check if the scan date has active setups.")


Running: Dynamic Improved + Accumulated Liquidity v2
Mode: Dynamic Improved + Accumulated Liquidity v2 diagnostics
Scan complete — tickers processed: 179 | Breaker+FVG hits (any candle): 189 | hits at candle 108 back: 10
Candle 108 back timestamp: 2026-04-01 09:15:00+05:30
Breaker+FVG signals

Candle: 108 back
Time: 2026-04-01 09:15:00+05:30

1. WIPRO.NS [+ISL Sweep @ 187.03] [+Base ISL Sweep @ 187.03] [Target ISH @ 196.90, 1.78 ATR]  (swept_lows: 4 | base_isl: 187.03 | low_hold: 22.5 | low_touches: 1 | low_vol: 0.71x | low_clusters: 4 | deeper_isls: 0 | target_highs: 6 | base_ish: 196.90 | target_hold: 76.8 | target_touches: 0 | target_vol: 1.11x | target_clusters: 6 | target_spacing_atr: 1.117 | ratio: 1.613 | atr_r: 1.214 | fvg/atr: 1.065 | score: 2.0855)
2. JUBLFOOD.NS [Target ISH @ 470.05, 3.29 ATR]  (swept_lows: 0 | base_isl: nan | low_hold: 0.0 | low_touches: 0 | low_vol: 0.00x | low_clusters: 0 | deeper_isls: 0 | target_highs: 4 | base_ish: 470.05 | target_hold: 13.0 | target_t